In [18]:
from src.language_models.utils import batchify, get_batch
from src.language_models.dictionary_corpus import Dictionary, Corpus
import pickle 
import torch

In [19]:
with open("test_orc_tokenized.pkl", "rb") as f:
    test_data = pickle.load(f)

In [20]:
test_data

tensor([  146, 27149,   539,  ...,   280,    62,    19])

In [21]:
test = batchify(test_data, 16, device = "cpu")

In [22]:
test.shape

torch.Size([676, 16])

In [29]:
with torch.no_grad():
    for i in range(0, test.size(0) - 1, 35):
        seq_len = min(35, test.size(0) - 1 - i)
        data = test[i : i + seq_len]
        targets = test[i + 1 : i + 1 + seq_len].reshape(-1)
        # Compare to get_batch output
        data_gb, targets_gb = get_batch(test, i, 35)
        print("Is targets_gb == targets from original batchified data?", torch.equal(targets_gb, targets))
        break

Is targets_gb == targets from original batchified data? True


Donc c'est shape seq_len, batch size

In [2]:
corpus = Corpus("data/orc_datasets/freq_8", save_tokenized=False)

In [5]:
corpus.valid

tensor([3100,   11,   42,  ..., 1285,   18,   19])

In [17]:
from src.language_models.model import RNNModel, Transformer

vocab_size = 50000  # Example vocab size
emsize = 650
nhid = 650
nlayers = 2
nheads = 10
d_model = 650
d_ff = 1300

rnn = RNNModel("LSTM", vocab_size, emsize, nhid, nlayers, dropout=0.5, tie_weights=False)
transformer = Transformer(vocab_size, d_model=d_model, nhead=nheads, num_layers=nlayers, dim_feedforward=d_ff, dropout=0.1)

rnn_params = sum(p.numel() for p in rnn.parameters())
transformer_params = sum(p.numel() for p in transformer.parameters())

print(f"RNNModel parameters: {rnn_params}")
print(f"Transformer parameters: {transformer_params}")

RNNModel parameters: 71820400
Transformer parameters: 72489900


In [36]:
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=50000,  # ton tokenizer
)

model = GPT2LMHeadModel(config)

In [32]:
model.parameters()

<generator object Module.parameters at 0x000001DBDC84E960>

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(
    model.parameters(),
    lr=5e-4,             # GPT-2 paper default
    betas=(0.9, 0.95),   # GPT-2 betas
    weight_decay=0.01
)

num_training_steps = len(train_data) // args.bptt * args.epochs
num_warmup_steps = 2000

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

In [1]:
import numpy as np

In [2]:
np.median([0.84,1.8,0.67,1.093,0.52])

np.float64(0.84)

In [ ]:
from pathlib import Path

train_path = Path("data/english_data/train.txt")
lines = train_path.read_text(encoding="utf-8").splitlines()

total_sentences = len(lines)
matching_sentences = sum(
    1
    for line in lines
    if "?" in line and any(word in line for word in ["Who", "what", "whom"])
)

percentage = (matching_sentences / total_sentences * 100) if total_sentences else 0.0

print(f"Total sentences: {total_sentences}")
print(f"Matching sentences: {matching_sentences}")
print(f"Percentage: {percentage:.2f}%")